In [ ]:
#Extracción de base de datos de eventos adversos de openFDA

In [6]:
#Importo las librerías que voy a necesitar
import requests
import pandas as pd
import time

In [12]:
#Defino la dirección web específica (endpoint)
BASE_URL = "https://api.fda.gov/drug/event.json"

In [13]:
#Defino los compuestos y el sitio de almacenaje
drugs = [
    "IBUPROFEN",
    "NAPROXEN",
    "ACETAMINOPHEN",
    "TRAMADOL"
]
all_records = []

In [14]:
#Código de extracción
for drug in drugs:

    print(f"\nDownloading data for {drug}")

    for skip in range(0, 1000, 100):

        params = {
            "search": f'patient.drug.medicinalproduct:"{drug}"',
            "limit": 100,
            "skip": skip
        }

        response = requests.get(BASE_URL, params=params)

        print(response.url)
        print("Status:", response.status_code)

        if response.status_code == 200:

            data = response.json()

            results = data.get("results", [])

            for result in results:

                result["searched_drug"] = drug

                all_records.append(result)

        else:
            print("Error downloading data")

        time.sleep(1)


https://api.fda.gov/drug/event.json?search=patient.drug.medicinalproduct%3A%22IBUPROFEN%22&limit=100&skip=0
Status: 200
https://api.fda.gov/drug/event.json?search=patient.drug.medicinalproduct%3A%22IBUPROFEN%22&limit=100&skip=100
Status: 200
https://api.fda.gov/drug/event.json?search=patient.drug.medicinalproduct%3A%22IBUPROFEN%22&limit=100&skip=200
Status: 200
https://api.fda.gov/drug/event.json?search=patient.drug.medicinalproduct%3A%22IBUPROFEN%22&limit=100&skip=300
Status: 200
https://api.fda.gov/drug/event.json?search=patient.drug.medicinalproduct%3A%22IBUPROFEN%22&limit=100&skip=400
Status: 200
https://api.fda.gov/drug/event.json?search=patient.drug.medicinalproduct%3A%22IBUPROFEN%22&limit=100&skip=500
Status: 200
https://api.fda.gov/drug/event.json?search=patient.drug.medicinalproduct%3A%22IBUPROFEN%22&limit=100&skip=600
Status: 200
https://api.fda.gov/drug/event.json?search=patient.drug.medicinalproduct%3A%22IBUPROFEN%22&limit=100&skip=700
Status: 200
https://api.fda.gov/drug/

In [15]:
#Verifico descarga
len(all_records)

4000

In [19]:
#Guardo JSON RAW
import json

with open("data_raw/fda_raw_4000.json", "w") as file:
    json.dump(all_records, file)

In [20]:
#Corroboro ubicación actual
import os

os.getcwd()

'C:\\Users\\luisi\\Desktop\\Bioinformatica\\Pharmacovigilance_proyect\\Data\\Scripts\\Notebooks'

In [24]:
#Para extraer los campos que son de interés para mi análisis y convertirlos en tabla:

import pandas as pd
import os 

#Crear carpeta
os.makedirs("data_processed", exist_ok=True)

#Lista staging vacía:
staging_records = []

# Recorrer registros raw
for record in all_records:

    patient = record.get("patient", {})

    reactions = patient.get("reaction", [])

    # Unnesting, Desanidar JSON
    # una reacción = una fila
    for reaction in reactions:

        staging_records.append({

            "report_id":
                record.get("safetyreportid"),

            "receive_date":
                record.get("receivedate"),

            "searched_drug":
                record.get("searched_drug"),

            "reaction":
                reaction.get(
                    "reactionmeddrapt", ""
                ),

            "serious":
                record.get("serious"),

            "death":
                record.get("seriousnessdeath"),

            "hospitalization":
                record.get(
                    "seriousnesshospitalization"
                ),

            "patient_sex":
                patient.get("patientsex"),

            "patient_age":
                patient.get("patientonsetage"),

            "patient_weight":
                patient.get("patientweight"),

            "country":
                record.get("occurcountry")
        })

# Crear dataframe (Tabla plana)
df_staging = pd.DataFrame(staging_records)

# Exportar CSV staging
df_staging.to_csv(
    "data_processed/fda_staging.csv",
    index=False
)

print("Staging table created successfully")

Staging table created successfully


In [25]:
#Data Cleaning
import pandas as pd

df = pd.read_csv(
    "data_processed/fda_staging.csv"
)
#Inpección inicial
df.head()

,report_id,receive_date,searched_drug,reaction,serious,death,hospitalization,patient_sex,patient_age,patient_weight,country
0,10003301,20140228,IBUPROFEN,Dyspepsia,1,NaN,NaN,2.0,NaN,NaN,NaN
1,10003301,20140228,IBUPROFEN,Renal impairment,1,NaN,NaN,2.0,NaN,NaN,NaN
2,10003319,20140312,IBUPROFEN,Cholecystectomy,1,NaN,1.0,2.0,46.0,NaN,US
3,10003319,20140312,IBUPROFEN,Nephrolithiasis,1,NaN,1.0,2.0,46.0,NaN,US
4,10003319,20140312,IBUPROFEN,Biliary tract disorder,1,NaN,1.0,2.0,46.0,NaN,US


In [26]:
#Data cleaning
#Ver estructura
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17223 entries, 0 to 17222
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   report_id        17223 non-null  int64  
 1   receive_date     17223 non-null  int64  
 2   searched_drug    17223 non-null  object 
 3   reaction         17223 non-null  object 
 4   serious          17223 non-null  int64  
 5   death            2242 non-null   float64
 6   hospitalization  7721 non-null   float64
 7   patient_sex      16985 non-null  float64
 8   patient_age      14397 non-null  float64
 9   patient_weight   8669 non-null   float64
 10  country          14190 non-null  object 
dtypes: float64(5), int64(3), object(3)
memory usage: 1.4+ MB


In [32]:
#Estadísticas rápidas
pd.set_option("display.float_format", "{:.2f}".format)
df.describe().round(2)

,report_id,receive_date,serious,death,hospitalization,patient_sex,patient_age,patient_weight
count,17223.00,17223.00,17223.00,2242.00,7721.00,16985.00,14397.00,8669.00
mean,10077819.54,20140340.56,1.13,1.09,1.01,1.65,104.46,80.45
std,61132.95,729.01,0.33,0.29,0.08,0.51,1093.42,26.24
min,10003301.00,20131121.00,1.00,1.00,1.00,0.00,1.00,0.66
25%,10035908.00,20140325.00,1.00,1.00,1.00,1.00,39.00,63.00
50%,10061738.00,20140404.00,1.00,1.00,1.00,2.00,53.00,77.00
75%,10094871.00,20140421.00,1.00,1.00,1.00,2.00,64.00,94.00
max,10261074.00,20140626.00,2.00,2.00,2.00,2.00,27839.00,379.00


In [29]:
#Analizar nulos
df.isnull().sum()

report_id              0
receive_date           0
searched_drug          0
reaction               0
serious                0
death              14981
hospitalization     9502
patient_sex          238
patient_age         2826
patient_weight      8554
country             3033
dtype: int64

In [34]:
#Homogeneizar texto
df["searched_drug"] = (
    df["searched_drug"]
    .str.upper()
)

df["reaction"] = (
    df["reaction"]
    .str.upper()
)

In [35]:
#Convertir sexo FDA a Male y Female
sex_map = {
    1: "MALE",
    2: "FEMALE"
}

df["patient_sex"] = (
    df["patient_sex"]
    .map(sex_map)
)

In [37]:
#Convertir fechas
df["receive_date"] = pd.to_datetime(
    df["receive_date"],
    format="%Y%m%d",
    errors="coerce"
)

In [38]:
#Convertir edades a valores numéricos 
df["patient_age"] = pd.to_numeric(
    df["patient_age"],
    errors="coerce"
)

In [40]:
#Detectar outliers
df[df["patient_age"] > 120]

,report_id,receive_date,searched_drug,reaction,serious,death,hospitalization,patient_sex,patient_age,patient_weight,country
1283,10032459,2014-03-24,IBUPROFEN,MALIGNANT NEOPLASM PROGRESSION,1,1.00,NaN,MALE,22241.00,85.30,AU
1284,10032459,2014-03-24,IBUPROFEN,ASTHENIA,1,1.00,NaN,MALE,22241.00,85.30,AU
1285,10032459,2014-03-24,IBUPROFEN,HOT FLUSH,1,1.00,NaN,MALE,22241.00,85.30,AU
2126,10051663,2014-04-01,IBUPROFEN,DIZZINESS,2,NaN,NaN,FEMALE,725.00,84.80,US
2127,10051663,2014-04-01,IBUPROFEN,DRUG HYPERSENSITIVITY,2,NaN,NaN,FEMALE,725.00,84.80,US
...,...,...,...,...,...,...,...,...,...,...,...
16350,10098383,2014-04-23,TRAMADOL,BLOOD PRESSURE INCREASED,1,NaN,1.00,FEMALE,773.00,74.40,US
16351,10098383,2014-04-23,TRAMADOL,DRUG INEFFECTIVE,1,NaN,1.00,FEMALE,773.00,74.40,US
16786,10136313,2014-04-29,TRAMADOL,ANAEMIA,1,NaN,1.00,FEMALE,24488.00,76.70,US
16787,10136313,2014-04-29,TRAMADOL,HIATUS HERNIA,1,NaN,1.00,FEMALE,24488.00,76.70,US


In [41]:
#Limpieza de edades:
df = df[
    (df["patient_age"].isna()) |
    (
        (df["patient_age"] >= 0) &
        (df["patient_age"] <= 120)
    )
]

In [42]:
#Limpieza de columnas de severidad:
severity_map = {
    1: "YES"
}

df["serious"] = (
    df["serious"]
    .map(severity_map)
    .fillna("NO")
)

df["death"] = (
    df["death"]
    .map(severity_map)
    .fillna("NO")
)

df["hospitalization"] = (
    df["hospitalization"]
    .map(severity_map)
    .fillna("NO")
)

In [43]:
#Revisar nuevamente nulos
df.isnull().sum()

report_id             0
receive_date          0
searched_drug         0
reaction              0
serious               0
death                 0
hospitalization       0
patient_sex         505
patient_age        2826
patient_weight     8528
country            3033
dtype: int64

In [50]:
#Estadísticas limpias
pd.set_option(
    "display.float_format",
    "{:.2f}".format
)

df[
    ["patient_age", "patient_weight"]
].describe().round(2)

,patient_age,patient_weight
count,14330.00,8628.00
mean,51.09,80.47
std,18.20,26.29
min,1.00,0.66
25%,39.00,63.00
50%,53.00,77.10
75%,64.00,94.35
max,98.00,379.00


In [45]:
#Revisar categorías
df["searched_drug"].value_counts()

searched_drug
NAPROXEN         4572
TRAMADOL         4477
IBUPROFEN        4400
ACETAMINOPHEN    3707
Name: count, dtype: int64

In [46]:
#Revisar reacciones
df["reaction"].value_counts().head(20)

reaction
TOXICITY TO VARIOUS AGENTS    249
PAIN                          240
HEADACHE                      215
NAUSEA                        208
VOMITING                      205
DRUG INEFFECTIVE              189
FATIGUE                       188
DYSPNOEA                      162
DIARRHOEA                     161
DRUG ABUSE                    156
DIZZINESS                     147
DRUG INTERACTION              145
CARDIAC DISORDER              144
ABDOMINAL PAIN                140
ARTHRALGIA                    134
PAIN IN EXTREMITY             130
DRUG HYPERSENSITIVITY         124
FALL                          108
MALAISE                       105
OVERDOSE                      103
Name: count, dtype: int64

In [47]:
#Revisar sexos
df["patient_sex"].value_counts()

patient_sex
FEMALE    11169
MALE       5482
Name: count, dtype: int64

In [48]:
#Revisar severidad
df["serious"].value_counts()

serious
YES    14973
NO      2183
Name: count, dtype: int64

In [51]:
#Convertir peso a numérico
df["patient_weight"] = pd.to_numeric(
    df["patient_weight"],
    errors="coerce"
)

In [52]:
#Limpieza de outliers de peso 
df = df[
    (df["patient_weight"].isna()) |
    (
        (df["patient_weight"] >= 2) &
        (df["patient_weight"] <= 250)
    )
]

In [ ]:
#Corroborar 
df[
    ["patient_age", "patient_weight"]
].describe().round(2)

In [54]:
#Exportar datos
import os

os.makedirs(
    "data_processed",
    exist_ok=True
)

In [55]:
df.to_csv(
    "data_processed/fda_clean.csv",
    index=False
)

In [56]:
print(
    "Clean dataset created successfully"
)

Clean dataset created successfully


In [4]:
# Transformar patient_age en grupos etarios

import pandas as pd

# Cargar dataset limpio
df = pd.read_csv(
    "data_cleaning/fda_clean.csv"
)

# Convertir edad a numérico
df["patient_age"] = pd.to_numeric(
    df["patient_age"],
    errors="coerce"
)

# Función para crear rangos etarios
def age_group(age):

    if pd.isna(age):
        return "UNKNOWN"

    elif age <= 17:
        return "PEDIATRIC"

    elif age <= 64:
        return "ADULT"

    else:
        return "GERIATRIC"

# Aplicar función
df["age_group"] = (
    df["patient_age"]
    .apply(age_group)
)

# Verificar resultado
print(
    df[
        ["patient_age", "age_group"]
    ].head(20)
)

# Guardar nuevamente el archivo actualizado
df.to_csv(
    "data_cleaning/fda_clean.csv",
    index=False
)

print(
    "age_group column added successfully"
)

    patient_age  age_group
0           NaN    UNKNOWN
1           NaN    UNKNOWN
2          46.0      ADULT
3          46.0      ADULT
4          46.0      ADULT
5          46.0      ADULT
6          18.0      ADULT
7          18.0      ADULT
8          33.0      ADULT
9          33.0      ADULT
10         54.0      ADULT
11         13.0  PEDIATRIC
12         13.0  PEDIATRIC
13         13.0  PEDIATRIC
14         13.0  PEDIATRIC
15         13.0  PEDIATRIC
16         13.0  PEDIATRIC
17         13.0  PEDIATRIC
18         13.0  PEDIATRIC
19         13.0  PEDIATRIC
age_group column added successfully
